In [80]:
import matplotlib.pyplot as plt
import numpy as np
import torch as tn
from igakit import cad
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from ttnte.xs.benchmarks import c5g7
from ttnte.iga import IGAMesh
from ttnte.sources import IsotropicInternalSource
from ttnte.assemblers import MatrixAssembler, TTAssembler
from ttnte.linalg import LinearOperator, eig

tn.set_default_dtype(tn.float64)
tn.set_num_threads(14)

In [81]:
# Discretization
num_ordinates = 1024

# Get XS data
xs_server = c5g7()

In [82]:
length = 6.5
li1 = cad.line(p0 = (-length/2,-length/2),p1=(length/2,-length/2))
li2 = cad.line(p0 = (-length/2,length/2),p1=(length/2,length/2))
patch = cad.ruled(li1,li2)
print("ruled patch",patch.points)

# Create source
source = IsotropicInternalSource()
# Source control points
source_ctrlpts = np.zeros((xs_server.num_groups, *patch.shape))
source_ctrlpts[0, ...] = 1# cm^{-1}sec^{-1} 
source_ctrlpts[1, :,:] = [[2,3],[4,5]]
source_ctrlpts[2, :,:] = [[1,1],[0,0]]

#add patch
source.add_patch(pid = 0, patch=patch, source_ctrlpts=source_ctrlpts)
print(source._patches[0])

#turn patches into a mesh
patches = {}
patches[patch]="UO2"
mesh = IGAMesh(patches)

for p in range(mesh.num_patches):
    mesh.refine(p, 3, 2)

mesh.connect(source=source)


ruled patch [[[-3.25 -3.25  0.  ]
  [-3.25  3.25  0.  ]]

 [[ 3.25 -3.25  0.  ]
  [ 3.25  3.25  0.  ]]]
(<igakit.nurbs.NURBS object at 0x765d746c6f60>, array([[[1., 1.],
        [1., 1.]],

       [[2., 3.],
        [4., 5.]],

       [[1., 1.],
        [0., 0.]],

       [[0., 0.],
        [0., 0.]],

       [[0., 0.],
        [0., 0.]],

       [[0., 0.],
        [0., 0.]],

       [[0., 0.],
        [0., 0.]]]))


In [83]:
#create list of coords to check the source at
coords = np.array([[0,1,0.5,0.5,0,1,0,1],[0,0,0,1,0.5,0.5,1,1]])
print(source._converted)
print(coords.ndim)
print(coords.shape[0])

#evaluate points at those coords
source_at_coords = source.evaluate_patch(pid=0, coords = coords)
print(source_at_coords[1])
print(source_at_coords[2])

True
2
2
[[[-3.25, -3.25, 0.0, 1.0], [-3.25, 3.25, 0.0, 1.0]], [[3.25, -3.25, 0.0, 1.0], [3.25, 3.25, 0.0, 1.0]]]
[2.  4.  3.  4.  2.5 4.5 3.  5. ]
[1.  0.  0.5 0.5 1.  0.  1.  0. ]


In [84]:
#test source on circle
origin= cad.line(p0=(0,0),p1=(0,0))
circle = cad.circle(radius=1, angle=np.pi/2.0)
print(circle.points)

quarter= cad.ruled(origin,circle)
print(quarter.points)

sourceb = IsotropicInternalSource()
source_ctrlptsb = np.zeros((xs_server.num_groups, *quarter.shape))
source_ctrlptsb[0,:,0] = 1
print(source_ctrlptsb)
sourceb.add_patch(pid=0, patch=quarter, source_ctrlpts=source_ctrlptsb)

#create mesh
patchesb = {}
patchesb[quarter] = "UO2"

meshb = IGAMesh(patches)

for p in range(meshb.num_patches):
    meshb.refine(p, 3, 2)

meshb.connect(source=sourceb)

[[1.00000000e+00 1.01465364e-17 0.00000000e+00]
 [1.00000000e+00 1.00000000e+00 0.00000000e+00]
 [1.99673462e-16 1.00000000e+00 0.00000000e+00]]
[[[0.00000000e+00 0.00000000e+00 0.00000000e+00]
  [1.00000000e+00 1.01465364e-17 0.00000000e+00]]

 [[0.00000000e+00 0.00000000e+00 0.00000000e+00]
  [1.00000000e+00 1.00000000e+00 0.00000000e+00]]

 [[0.00000000e+00 0.00000000e+00 0.00000000e+00]
  [1.99673462e-16 1.00000000e+00 0.00000000e+00]]]
[[[1. 0.]
  [1. 0.]
  [1. 0.]]

 [[0. 0.]
  [0. 0.]
  [0. 0.]]

 [[0. 0.]
  [0. 0.]
  [0. 0.]]

 [[0. 0.]
  [0. 0.]
  [0. 0.]]

 [[0. 0.]
  [0. 0.]
  [0. 0.]]

 [[0. 0.]
  [0. 0.]
  [0. 0.]]

 [[0. 0.]
  [0. 0.]
  [0. 0.]]]


In [87]:
#create list of coords to check the source at
coords = np.array([[0,1,0.5,0.5,0.5,0,1,0,1],[0,0,0,1,0.5,0.5,0.5,1,1]])

#evaluate points at those coords
source_at_coords = sourceb.evaluate_patch(pid=0, coords = coords)
print(source_at_coords[0])

[[[0.0, 0.0, 0.0, 1.0], [1.0, 1.0146536357569526e-17, 0.0, 1.0]], [[0.0, 0.0, 0.0, 1.0], [0.7071067811865476, 0.7071067811865475, 0.0, 0.7071067811865476]], [[0.0, 0.0, 0.0, 1.0], [1.9967346175427393e-16, 1.0, 0.0, 1.0]]]
[1.         1.         1.         0.         0.53950429 0.5
 0.5        0.         0.        ]
